# EMCAD experiments
Choose baseline, positive_dice, positive_dice_full_train, local, luma, stride4, stride2_rgb or dct_aux.
Mixed-frame training, natural DCT, verified originals and original-resolution development are built in.
The last cell starts training.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_experiment_config

experiment = 'baseline'  # JPEG ablation: dct576, rgb576, jpeg576, jpeg576_pretrained
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1024, 1024) if cfg.model.forensic_mode == 'jpeg' else None
with torch.device('meta'):
    budget_model = build_model(cfg.model, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          use_valid_mask=cfg.dataset.resize_mode == 'letterbox', native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
